<a href="https://colab.research.google.com/github/anhelus/alsia-workshop/blob/master/Notebooks/02_ai_assisted_labeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Setup
!git clone https://github.com/anhelus/alsia-workshop.git
!mv alsia-workshop/Notebooks/* .

Cloning into 'alsia-workshop'...
remote: Enumerating objects: 116, done.
remote: Counting objects: 100% (42/42), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 116 (delta 5), reused 23 (delta 1), pack-reused 74 (from 2)
Receiving objects: 100% (116/116), 75.23 MiB | 31.74 MiB/s, done.
Resolving deltas: 100% (21/21), done.


# AI-assisted Labeling
Benvenuti nel notebook di accompagnamento per il talk AI-Assisted Labeling! In questo notebook, vedremo insieme come configurare il nostro ambiente di sviluppo, effettuare una pre-annotazione semantica con modelli fondazionali e importare questi dati in Label Studio per una rifinitura manuale.

## 1. Setup
Come primo passo abbiamo bisogno di importare le librerie di cui abbiamo bisogno per l'esecuzione dei nostri codici. Come in precedenza, useremo la libreria di object detectors _Ultralytics_, che andiamo ad installare:

In [2]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 27.3 MB/s eta 0:00:00


Successivamente, importiamo dalla libreria il modello che andremo ad utilizzare, ovvero YOLO World:

In [3]:
from ultralytics import YOLOWorld

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


## 2. Pre-Annotazione con modelli fondazionali

### Predizione zero-shot

Come abbiamo visto nelle presentazioni precedenti, possiamo andare ad utilizzare un open-world object detector per produrre delle pre-annotazioni, constentendoci di effettuare la fase di annotazione manuale in maniera molto più efficiente, riducendola ad una semplice rifinitura e controllo delle bounding box predette dal modello.

In pratica andiamo a:
1. Creiamo un'istanza di YOLOWorld
2. Impostiamo un insieme di prompt (seguendo la tecnica del _prompt ensembling_)
3. Eseguiamo la predizione sulle immagini

In [4]:
open_model = YOLOWorld("models/yolov8s-world.pt")
open_model.set_classes([
    "a lettuce plant",
    "a photo of lettuce",
    "a head of lettuce",
    "a lettuce seedling",
    "a young lettuce plant",
    "a small green plant",
    "lettuce growing in soil",
    "a photo of lettuce from above",
    "a rosette of green leaves",
    "a leafy green vegetable",
    "a crop of lettuce"
])
results = open_model.predict(source="data/lettuce", conf=0.05, save_txt=True)

requirements: Ultralytics requirement ['git+https://github.com/ultralytics/CLIP.git'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 18 packages in 750ms
Prepared 2 packages in 2.29s
Installed 2 packages in 1ms
 + clip==1.0 (from git+https://github.com/ultralytics/CLIP.git@81ff68ed7ffcac3b40484c914f104f816757308d)
 + ftfy==6.3.1

requirements: AutoUpdate success ✅ 3.8s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect



100%|████████████████████████████████████████| 338M/338M [00:01<00:00, 183MiB/s]



image 1/8 /content/data/lettuce/lettuce-0.png: 384x640 3 a small green plants, 2 a leafy green vegetables, 2 a crop of lettuces, 699.6ms
image 2/8 /content/data/lettuce/lettuce-1.png: 384x640 1 a small green plant, 443.3ms
image 3/8 /content/data/lettuce/lettuce-2.png: 384x640 2 a small green plants, 1 a photo of lettuce from above, 1 a crop of lettuce, 497.1ms
image 4/8 /content/data/lettuce/lettuce-3.png: 384x640 (no detections), 467.8ms
image 5/8 /content/data/lettuce/lettuce-4.png: 384x640 (no detections), 502.4ms
image 6/8 /content/data/lettuce/lettuce-5.png: 384x640 4 a crop of lettuces, 466.7ms
image 7/8 /content/data/lettuce/lettuce-6.png: 384x640 1 a small green plant, 4 a crop of lettuces, 476.4ms
image 8/8 /content/data/lettuce/lettuce-7.png: 384x640 2 a crop of lettuces, 484.6ms
Speed: 4.4ms preprocess, 504.7ms inference, 4.5ms postprocess per image at shape (1, 3, 384, 640)
Results saved to /content/runs/detect/predict
6 labels saved to /content/runs/detect/predict/labels

### Salvataggio dei risultati

Le immagini annotate che abbiamo visualizzato finora non sono adatte per trasferire le predizioni su altri tool. Pertanto, con il parametro `save_txt` chiediamo ad Ultralytics di salvare i risultati della detection in dei file leggibili da altri modelli e/o software che permettano di elaborare questi dati.

Usando `save_txt` otteniamo dei file di testo nel cosiddetto _formato YOLO_. Grazie ad un processo di conversione possiamo trasferire questi risultati su software di etichettatura come Label Studio o LabelMe. Per questa esercitazione, useremo Label Studio!